In [ ]:
"""
NBFC Enterprise Risk Architecture & Data Pipeline
-------------------------------------------------
Orchestration layer: delegates all logic to src/pipeline.py so the
notebook stays thin and every calculation remains covered by the test suite.
"""

import sys, os

# Make src/ importable when running in Google Colab
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')) if '__file__' not in dir() else os.path.dirname(os.path.abspath(__file__)), '..'))

import warnings
warnings.filterwarnings('ignore')

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

from src.pipeline import generate_portfolio, extract_hitlist, save_to_database

print("INITIATING END-TO-END NBFC DATA PIPELINE...")

print("\n[1/4] Synthesizing Demographic and Asset Data...")
print("[2/4] Executing NPA Probability Algorithm...")
df_master = generate_portfolio(num_records=1500, seed=42)

print("[3/4] Architecting SQL Database and Exporting Master Portfolio...")
df_master.to_csv('nbfc_master_portfolio.csv', index=False)

print("[4/4] Filtering Target Accounts for Field Operations...")
df_hitlist = extract_hitlist(df_master)
df_hitlist.to_csv('nbfc_operational_hitlist.csv', index=False)

save_to_database(df_master, df_hitlist, 'nbfc_enterprise_system.db')

print("\n--- PIPELINE COMPLETE ---")
print(f"Total Master Records: {len(df_master)}")
print(f"Total High-Risk Hitlist Records: {len(df_hitlist)}")

if IN_COLAB:
    print('\nTriggering file downloads...')
    files.download('nbfc_enterprise_system.db')
    files.download('nbfc_master_portfolio.csv')
    files.download('nbfc_operational_hitlist.csv')
